## Statistical Analysis

Correlation, hypothesis testing, regression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/cleaned_retail_data.csv')
active = df[df['IsCancelled']==False].copy()
print(active.shape)

In [ ]:
# correlation matrix
cols = ['Quantity','UnitPrice','TotalRevenue','AvgOrderValue','Recency','Frequency','Monetary']
corr = active[cols].corr()

plt.figure(figsize=(9,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# hypothesis test 1 - UK vs International revenue per transaction
uk = active[active['IsUK']=='UK']['TotalRevenue']
intl = active[active['IsUK']=='International']['TotalRevenue']

stat, p = stats.mannwhitneyu(uk, intl, alternative='two-sided')
print('UK mean:', uk.mean().round(2))
print('International mean:', intl.mean().round(2))
print('p-value:', round(p, 5))
if p < 0.05:
    print('Significant difference exists between UK and International transactions')
else:
    print('No significant difference')

In [ ]:
# hypothesis test 2 - weekday vs weekend revenue
weekdays = active[active['DayOfWeek'].isin(['Monday','Tuesday','Wednesday','Thursday','Friday'])]['TotalRevenue']
weekends = active[active['DayOfWeek'].isin(['Saturday','Sunday'])]['TotalRevenue']

stat2, p2 = stats.mannwhitneyu(weekdays, weekends, alternative='two-sided')
print('Weekday mean:', weekdays.mean().round(2))
print('Weekend mean:', weekends.mean().round(2))
print('p-value:', round(p2, 5))

In [ ]:
# quarterly comparison
q = active.groupby('Quarter')['TotalRevenue'].agg(['sum','mean','count']).round(2)
q.columns = ['Total Revenue','Avg per txn','Count']
print(q)

q['Total Revenue'].plot(kind='bar', color=['green','blue','orange','red'])
plt.title('Revenue by Quarter')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# rfm distributions
rfm = active[active['CustomerID']!='Guest'].drop_duplicates('CustomerID')

fig, axes = plt.subplots(1, 3, figsize=(14,4))
axes[0].hist(rfm['Recency'].dropna(), bins=40, color='red')
axes[0].set_title('Recency')
axes[1].hist(rfm['Frequency'].dropna().clip(upper=50), bins=40, color='blue')
axes[1].set_title('Frequency')
axes[2].hist(rfm['Monetary'].dropna().clip(upper=5000), bins=40, color='green')
axes[2].set_title('Monetary')
plt.tight_layout()
plt.show()

In [ ]:
# simple regression - does hour predict avg revenue?
hourly = active.groupby('Hour')['TotalRevenue'].mean().reset_index()
slope, intercept, r, p, se = stats.linregress(hourly['Hour'], hourly['TotalRevenue'])

plt.figure(figsize=(9,4))
plt.scatter(hourly['Hour'], hourly['TotalRevenue'])
plt.plot(hourly['Hour'], slope*hourly['Hour']+intercept, color='red', label=f'R²={r**2:.3f}')
plt.title('Hour vs Avg Revenue')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Revenue (£)')
plt.legend()
plt.tight_layout()
plt.show()
print('R²:', round(r**2, 4), '| p-value:', round(p, 5))